# Solver guardrail simulation

This notebook proves that the **real packing solver changes its decision when a packing guardrail applies**.

Every scenario has the same structure:

**1. Config → 2. Items → 3. Boxes → `solve_order()` → observable solver decision**

We do not manufacture invalid packing plans and we do not call the validator directly. Each scenario runs through the public solver API in both **Fast** and **Best** modes.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from bin_packing_3d import solve_order
from bin_packing_3d.models import PackingError

def run_both(config, items, boxes):
    """Run the same three inputs through both public solver modes."""
    order = {"OrderId": "SIM", "OrderNo": "SIM", "Items": items}
    results = {}
    for mode in ("fast", "best"):
        try:
            result = solve_order(order, boxes, mode=mode, logs=True, **config)
            selected = [box.box_code for box in result.packed_boxes]
            print(f"\n{mode.upper()} RESULT → selected {selected}; validation={result.validation.valid}")
            results[mode] = result
        except PackingError as exc:
            print(f"\n{mode.upper()} RESULT → REJECTED: {type(exc).__name__}: {exc}")
            results[mode] = exc
    return results

def selected_codes(result):
    return [box.box_code for box in result.packed_boxes]


## Scenario 1 — Weight limit makes the solver skip the smaller carton

The item geometrically fits both cartons. **SmallBox is rejected only because its 5 kg weight limit is below the item's 6 kg weight.** The solver must therefore choose HeavyBox.


In [2]:
# 1. CONFIG
config = {
    "bin_buffer": {"length": 0, "width": 0, "height": 0},
    "max_fill_pct": 100,
}

# 2. ITEMS
items = [{
    "Code": "HeavyItem", "Length": 10, "Width": 10, "Height": 10,
    "Weight": 6, "Quantity": 1, "VerticalRotation": 0,
}]

# 3. BOXES
boxes = [
    {"Code": "SmallBox", "Length": 12, "Width": 12, "Height": 12, "MaxWeight": 5},
    {"Code": "HeavyBox", "Length": 14, "Width": 14, "Height": 14, "MaxWeight": 10},
]

results = run_both(config, items, boxes)
assert all(selected_codes(results[m]) == ["HeavyBox"] for m in ("fast", "best"))
print("\nGUARDRAIL PASS → SmallBox skipped because of MaxWeight.")


=== 3D packing log ===

Order summary
Metric                 | Value     
-----------------------+-----------
Physical items         | 1         
Total item volume      | 1,000 mm^3
Total item weight      | 6 kg      
Effective maximum fill | 100%      

Candidate cartons (smallest external volume first)
Carton   | External mm^3 | Usable L x W x H (mm) | Fill cap mm^3 | Result                            
---------+---------------+-----------------------+---------------+-----------------------------------
SmallBox | 1,728         | 12 x 12 x 12          | 1,728         | REJECTED by weight: 6 > 5 kg      
HeavyBox | 2,744         | 14 x 14 x 14          | 2,744         | SELECTED: smallest feasible carton

Orientation checks for HeavyBox
L x W x H (mm) | Result
---------------+-------
10 x 10 x 10   | FITS  

3D search decisions
Item        | Carton attempted    | Outcome  | Why                                     
------------+---------------------+----------+--------------------------


3D search decisions
Selected strategy candidate.

Final validated assignments
Item        | Assigned carton     | Packed L x W x H (mm) | XYZ position (mm) | Carton result         
------------+---------------------+-----------------------+-------------------+-----------------------
HeavyItem#1 | carton-1 (HeavyBox) | 10 x 10 x 10          | (0, 0, 0)         | 36.443149% carton fill

Final decision
Selected carton(s): HeavyBox
Not selected: SmallBox. Passing the initial screen does not prove a complete 3D packing; larger alternatives are also left unused once the strategy has a complete plan.

BEST RESULT → selected ['HeavyBox']; validation=True

GUARDRAIL PASS → SmallBox skipped because of MaxWeight.


## Scenario 2 — 6 mm carton buffer makes the solver choose the larger carton

The 10 mm-high item physically fits TightBox's external 15 mm height. With the configured **6 mm height buffer**, TightBox has only **9 mm usable height**, so it must be skipped. RoomyBox remains feasible.


In [3]:
# 1. CONFIG
config = {
    "bin_buffer": {"length": 0, "width": 0, "height": 6},
    "max_fill_pct": 100,
}

# 2. ITEMS
items = [{
    "Code": "BufferedItem", "Length": 10, "Width": 10, "Height": 10,
    "Weight": 1, "Quantity": 1, "VerticalRotation": 0,
}]

# 3. BOXES
boxes = [
    {"Code": "TightBox", "Length": 12, "Width": 12, "Height": 15, "MaxWeight": 10},
    {"Code": "RoomyBox", "Length": 14, "Width": 14, "Height": 20, "MaxWeight": 10},
]

results = run_both(config, items, boxes)
assert all(selected_codes(results[m]) == ["RoomyBox"] for m in ("fast", "best"))
print("\nGUARDRAIL PASS → TightBox skipped because 15 - 6 = 9 mm usable height.")


=== 3D packing log ===

Order summary
Metric                 | Value     
-----------------------+-----------
Physical items         | 1         
Total item volume      | 1,000 mm^3
Total item weight      | 1 kg      
Effective maximum fill | 100%      

Candidate cartons (smallest external volume first)
Carton   | External mm^3 | Usable L x W x H (mm) | Fill cap mm^3 | Result                                           
---------+---------------+-----------------------+---------------+--------------------------------------------------
TightBox | 2,160         | 12 x 12 x 9           | 1,296         | REJECTED by geometry: no allowed orientation fits
RoomyBox | 3,920         | 14 x 14 x 14          | 2,744         | SELECTED: smallest feasible carton               

Orientation checks for TightBox
L x W x H (mm) | Result            
---------------+-------------------
10 x 10 x 10   | REJECTED: H 10 > 9

Orientation checks for RoomyBox
L x W x H (mm) | Result
---------------+-------
10 x

## Scenario 3 — Maximum fill percentage makes the solver skip a carton that physically fits

The item volume is **1,000 mm³**. SmallBox has **1,331 mm³** usable volume, so the item physically fits, but it occupies about **75.1%**. With `max_fill_pct = 70`, SmallBox must be skipped and LargeBox selected.


In [4]:
# 1. CONFIG
config = {
    "bin_buffer": {"length": 0, "width": 0, "height": 0},
    "max_fill_pct": 70,
}

# 2. ITEMS
items = [{
    "Code": "FillItem", "Length": 10, "Width": 10, "Height": 10,
    "Weight": 1, "Quantity": 1, "VerticalRotation": 0,
}]

# 3. BOXES
boxes = [
    {"Code": "SmallBox", "Length": 11, "Width": 11, "Height": 11, "MaxWeight": 10},
    {"Code": "LargeBox", "Length": 12, "Width": 12, "Height": 12, "MaxWeight": 10},
]

results = run_both(config, items, boxes)
assert all(selected_codes(results[m]) == ["LargeBox"] for m in ("fast", "best"))
print("\nGUARDRAIL PASS → SmallBox skipped because ~75.1% > 70% maximum fill.")


=== 3D packing log ===

Order summary
Metric                 | Value     
-----------------------+-----------
Physical items         | 1         
Total item volume      | 1,000 mm^3
Total item weight      | 1 kg      
Effective maximum fill | 70%       

Candidate cartons (smallest external volume first)
Carton   | External mm^3 | Usable L x W x H (mm) | Fill cap mm^3 | Result                                
---------+---------------+-----------------------+---------------+---------------------------------------
SmallBox | 1,331         | 11 x 11 x 11          | 931.7         | REJECTED by volume: 1,000 > 931.7 mm^3
LargeBox | 1,728         | 12 x 12 x 12          | 1,209.6       | SELECTED: smallest feasible carton    

Orientation checks for LargeBox
L x W x H (mm) | Result
---------------+-------
10 x 10 x 10   | FITS  

3D search decisions
Item       | Carton attempted    | Outcome  | Why                                           
-----------+---------------------+----------+------

## Scenario 4 — More than 6 items activates the stricter 70% fill rule

Seven 10 mm cubes can geometrically fit in a 20 × 20 × 20 mm carton. Their combined volume is **7,000 mm³**, or **87.5%** of SmallBox. Because the order has **7 physical items**, the configured high-item-count rule reduces the allowed fill to **70%**, forcing the solver to LargeBox.


In [5]:
# 1. CONFIG
config = {
    "bin_buffer": {"length": 0, "width": 0, "height": 0},
    "max_fill_pct": 100,
    "high_item_count_threshold": 6,
    "high_item_count_max_fill_pct": 70,
}

# 2. ITEMS
items = [{
    "Code": "Cube", "Length": 10, "Width": 10, "Height": 10,
    "Weight": 0.5, "Quantity": 7, "VerticalRotation": 0,
}]

# 3. BOXES
boxes = [
    {"Code": "SmallBox", "Length": 20, "Width": 20, "Height": 20, "MaxWeight": 10},
    {"Code": "LargeBox", "Length": 22, "Width": 22, "Height": 22, "MaxWeight": 10},
]

results = run_both(config, items, boxes)
assert all(selected_codes(results[m]) == ["LargeBox"] for m in ("fast", "best"))
print("\nGUARDRAIL PASS → 7 items activated the 70% rule, so SmallBox was skipped.")


=== 3D packing log ===

Order summary
Metric                 | Value     
-----------------------+-----------
Physical items         | 7         
Total item volume      | 7,000 mm^3
Total item weight      | 3.5 kg    
Effective maximum fill | 70%       

Candidate cartons (smallest external volume first)
Carton   | External mm^3 | Usable L x W x H (mm) | Fill cap mm^3 | Result                                   
---------+---------------+-----------------------+---------------+------------------------------------------
SmallBox | 8,000         | 20 x 20 x 20          | 5,600         | REJECTED by volume: 7,000 > 5,600 mm^3   
LargeBox | 10,648        | 22 x 22 x 22          | 7,453.6       | PASSED screen; evaluated during 3D search

3D search decisions
Selected fixed_carton_candidate. Best mode found a feasible fixed-carton combination with fewer cartons.

Final validated assignments
Item   | Assigned carton     | Packed L x W x H (mm) | XYZ position (mm) | Carton result         
-----

## Scenario 5 — Vertical rotation setting changes which carton is usable

The item is **30 × 10 × 20 mm**. TightBox is **20 × 30 × 10 mm**. It fits TightBox only if the item is allowed to turn vertically into **20 × 30 × 10 mm**.

Here `VerticalRotation = 0`, so TightBox must be skipped and FallbackBox used.


In [6]:
# 1. CONFIG
config = {
    "bin_buffer": {"length": 0, "width": 0, "height": 0},
    "max_fill_pct": 100,
}

# 2. ITEMS
items = [{
    "Code": "RotationItem", "Length": 30, "Width": 10, "Height": 20,
    "Weight": 1, "Quantity": 1, "VerticalRotation": 0,
}]

# 3. BOXES
boxes = [
    {"Code": "TightBox", "Length": 20, "Width": 30, "Height": 10, "MaxWeight": 10},
    {"Code": "FallbackBox", "Length": 35, "Width": 35, "Height": 25, "MaxWeight": 10},
]

results = run_both(config, items, boxes)
assert all(selected_codes(results[m]) == ["FallbackBox"] for m in ("fast", "best"))
print("\nGUARDRAIL PASS → TightBox skipped because VerticalRotation=0.")


=== 3D packing log ===

Order summary
Metric                 | Value     
-----------------------+-----------
Physical items         | 1         
Total item volume      | 6,000 mm^3
Total item weight      | 1 kg      
Effective maximum fill | 100%      

Candidate cartons (smallest external volume first)
Carton      | External mm^3 | Usable L x W x H (mm) | Fill cap mm^3 | Result                                           
------------+---------------+-----------------------+---------------+--------------------------------------------------
TightBox    | 6,000         | 20 x 30 x 10          | 6,000         | REJECTED by geometry: no allowed orientation fits
FallbackBox | 30,625        | 35 x 35 x 25          | 30,625        | SELECTED: smallest feasible carton               

Orientation checks for TightBox
L x W x H (mm) | Result                        
---------------+-------------------------------
30 x 10 x 20   | REJECTED: L 30 > 20, H 20 > 10
10 x 30 x 20   | REJECTED: H 20 > 10 

## Scenario 6 — No carton satisfies the weight guardrail, so the order fails

This time there is no fallback carton. The 11 kg item fits geometrically, but **every available carton has a lower maximum weight**. The public solver API must reject the order before returning a packing plan.


In [7]:
# 1. CONFIG
config = {
    "bin_buffer": {"length": 0, "width": 0, "height": 0},
    "max_fill_pct": 100,
}

# 2. ITEMS
items = [{
    "Code": "TooHeavy", "Length": 10, "Width": 10, "Height": 10,
    "Weight": 11, "Quantity": 1, "VerticalRotation": 0,
}]

# 3. BOXES
boxes = [
    {"Code": "BoxA", "Length": 20, "Width": 20, "Height": 20, "MaxWeight": 5},
    {"Code": "BoxB", "Length": 30, "Width": 30, "Height": 30, "MaxWeight": 10},
]

results = run_both(config, items, boxes)
assert all(isinstance(results[m], PackingError) for m in ("fast", "best"))
print("\nGUARDRAIL PASS → no carton can satisfy the weight rule, so the order was rejected.")



FAST RESULT → REJECTED: UnpackableItemError: No carton can contain physical item(s): TooHeavy#1

BEST RESULT → REJECTED: UnpackableItemError: No carton can contain physical item(s): TooHeavy#1

GUARDRAIL PASS → no carton can satisfy the weight rule, so the order was rejected.


## Final result

Each proof above went through **`solve_order()`** using only the scenario's **Config + Items + Boxes**. The notebook passes only if both Fast and Best exhibit the expected solver behaviour.


In [8]:
print("ALL SOLVER GUARDRAILS PASS")
print("6/6 scenarios produced the expected skip, fallback, or rejection in both Fast and Best modes.")


ALL SOLVER GUARDRAILS PASS
6/6 scenarios produced the expected skip, fallback, or rejection in both Fast and Best modes.
